# Flan-T5 · Fine-tuning on WikiSQL

Fine-tunes `google/flan-t5-base` for text-to-SQL generation on the [WikiSQL](https://github.com/salesforce/WikiSQL) dataset.

**Run this notebook first.** Its output checkpoint is the starting point for the companion `Flan-T5-Spider.ipynb` notebook, which continues training on the harder [Spider](https://yale-lily.github.io/spider) dataset.

**Environment:** Built for **Kaggle** (GPU T4 x2 accelerator). Uses `/kaggle/working/` for outputs.
To run elsewhere (Colab / local), change `WIKISQL_FINAL_DIR` and the other `/kaggle/...` paths, and make sure a CUDA GPU is available.

## Pipeline
1. Download & parse the raw WikiSQL data
2. Build SQL generation / repair helpers (the raw WikiSQL annotations aren't valid SQL strings on their own)
3. Tokenize with the Flan-T5 tokenizer
4. Fine-tune with `Seq2SeqTrainer`
5. Evaluate exact-match accuracy on the WikiSQL test split


In [ ]:
import subprocess
subprocess.run(
    ["pip", "install", "-q", "-U",
     "transformers", "datasets", "evaluate", "gradio",
     "accelerate", "sentencepiece", "pandas"],
    check=True,
)

import os, re, json, math, random, sqlite3, urllib.request, tarfile
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, Dataset
try:
    from IPython.display import display
except ImportError:
    display = print
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

os.environ["WANDB_DISABLED"]         = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "google/flan-t5-base"

# All outputs go to /kaggle/working/ -- downloadable after the session
WIKISQL_FINAL_DIR = "/kaggle/working/wikisql-flan-t5-large"

MAX_INPUT_LENGTH  = 256
MAX_TARGET_LENGTH = 128

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. In Kaggle: Settings -> Accelerator -> GPU T4 x2"
    )

device = "cuda"
n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus} x {torch.cuda.get_device_name(0)}")


## 1. Download and parse WikiSQL
Downloads the WikiSQL archive from the official GitHub repo and converts each split into a Hugging Face `Dataset`.

In [ ]:
WIKISQL_DIR = "/tmp/wikisql"


def _download_wikisql(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    if os.path.isdir(os.path.join(data_dir, "data")):
        print("WikiSQL already downloaded, skipping.")
        return
    archive = os.path.join(data_dir, "data.tar.bz2")
    print("Downloading WikiSQL (~9 MB) ...")
    urllib.request.urlretrieve(
        "https://github.com/salesforce/WikiSQL/raw/master/data.tar.bz2",
        archive,
    )
    with tarfile.open(archive, "r:bz2") as tar:
        try:
            tar.extractall(data_dir, filter="data")
        except TypeError:
            tar.extractall(data_dir)
    print("Done.")


def _parse_split(split_jsonl, tables_jsonl):
    tables = {}
    with open(tables_jsonl, encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)
            tables[t["id"]] = t
    examples = []
    with open(split_jsonl, encoding="utf-8") as f:
        for line in f:
            ex        = json.loads(line)
            table     = tables[ex["table_id"]]
            raw_conds = ex["sql"].get("conds") or []
            examples.append({
                "question": ex["question"],
                "table": {
                    "id":         table["id"],
                    "header":     [str(h) for h in table["header"]],
                    "types":      table.get("types", ["text"] * len(table["header"])),
                    "rows":       [],
                    "name":       table.get("name", table["id"]),
                    "page_title": table.get("page_title", ""),
                },
                "sql": {
                    "human_readable": "",
                    "sel":  int(ex["sql"]["sel"]),
                    "agg":  int(ex["sql"]["agg"]),
                    "conds": {
                        "column_index":   [int(c[0]) for c in raw_conds],
                        "operator_index": [int(c[1]) for c in raw_conds],
                        "condition":      [str(c[2]) for c in raw_conds],
                    },
                },
            })
    return Dataset.from_list(examples)


def load_wikisql_robust():
    _download_wikisql(WIKISQL_DIR)
    d = os.path.join(WIKISQL_DIR, "data")
    return DatasetDict({
        "train":      _parse_split(f"{d}/train.jsonl", f"{d}/train.tables.jsonl"),
        "validation": _parse_split(f"{d}/dev.jsonl",   f"{d}/dev.tables.jsonl"),
        "test":       _parse_split(f"{d}/test.jsonl",  f"{d}/test.tables.jsonl"),
    })


raw_datasets = load_wikisql_robust()
print(raw_datasets)

## 2. SQL generation & repair helpers
WikiSQL stores queries as (column, aggregation, conditions) triples rather than SQL text, so these helpers render them to SQL strings for training, and `repair_sql` post-processes model output to fix common formatting mistakes at inference time.

In [ ]:
import difflib

AGG_OPS  = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<", "OP", "!=", "LIKE"]
_AGG_SET = {a for a in AGG_OPS if a}
_OP_SET  = {o for o in COND_OPS if o and o != "OP"}


def normalize_type(type_name):
    t = str(type_name).strip().lower()
    return "number" if t in {"number","real","integer","int","float","numeric"} else "text"


def sanitize_table_name(table):
    name = table.get("name") or table.get("page_title") or table.get("id", "table")
    name = re.sub(r"[^A-Za-z0-9_]", "_", str(name)).strip("_")
    return name or "table"


def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def iter_conditions(sql_dict):
    conds = sql_dict.get("conds", {})
    if isinstance(conds, dict):
        yield from zip(
            conds.get("column_index",   []),
            conds.get("operator_index", []),
            conds.get("condition",      []),
        )
    elif isinstance(conds, list):
        yield from conds


def example_to_input(example):
    table   = example["table"]
    headers = list(table["header"])
    types   = list(table.get("types", ["text"] * len(headers)))
    cols    = ", ".join(f"{h} : {normalize_type(t)}" for h, t in zip(headers, types))
    tname   = sanitize_table_name(table)
    return (f"translate to SQL: question: {example['question']} "
            f"| table: {tname} | columns: {cols}")


def example_to_sql(example):
    table   = example["table"]
    headers = list(table["header"])
    sql     = example["sql"]
    sel_col = headers[sql["sel"]]
    agg     = AGG_OPS[sql["agg"]]
    select  = f"{agg}({quote_identifier(sel_col)})" if agg else quote_identifier(sel_col)
    tname   = sanitize_table_name(table)
    query   = f"SELECT {select} FROM {quote_identifier(tname)}"
    conds   = list(iter_conditions(sql))
    if conds:
        parts = [f"{quote_identifier(headers[ci])} {COND_OPS[oi]} '{v}'"
                 for ci, oi, v in conds]
        query += " WHERE " + " AND ".join(parts)
    return query


# SQL repair helpers
def _strip_quotes(s):
    s = (s or "").strip()
    for q in ('"', "'", "`"):
        if len(s) >= 2 and s.startswith(q) and s.endswith(q):
            return s[1:-1]
    return s


def _snap_column(name, headers):
    name = _strip_quotes(name)
    if not name: return headers[0]
    if name in headers: return name
    low = {h.lower(): h for h in headers}
    if name.lower() in low: return low[name.lower()]
    m = difflib.get_close_matches(name, list(headers), n=1, cutoff=0.0)
    return m[0] if m else headers[0]


def _snap_agg(t):
    t = (t or "").strip().upper()
    return t if t in _AGG_SET else ""


def _snap_op(t):
    t = (t or "").strip()
    if t == "<>": return "!="
    return t if t in _OP_SET else "="


def repair_sql(sql, headers, types, table_name):
    if not sql: return sql
    s = sql.strip().rstrip(";")
    head = re.match(
        r"^\s*SELECT\s+(.+?)\s+FROM\s+\S+(?:\s+WHERE\s+(.*))?$",
        s, re.IGNORECASE | re.DOTALL,
    )
    if not head: return sql
    sp = head.group(1).strip()
    wp = (head.group(2) or "").strip()
    agg = ""
    sel_raw = sp
    am = re.match(r"^(MAX|MIN|COUNT|SUM|AVG)\s*\(\s*(.+?)\s*\)$", sp, re.IGNORECASE)
    if am:
        agg     = am.group(1).upper()
        sel_raw = am.group(2)
    sel  = _snap_column(sel_raw, headers)
    sel_expr = f"{agg}({quote_identifier(sel)})" if agg else quote_identifier(sel)
    out  = f"SELECT {sel_expr} FROM {quote_identifier(table_name)}"
    if wp:
        conds = []
        for part in re.split(r"\s+AND\s+", wp, flags=re.IGNORECASE):
            cm = re.match(
                r"^\s*(.+?)\s*(<>|!=|>=|<=|=|>|<|LIKE)\s*(.+?)\s*$",
                part, re.IGNORECASE,
            )
            if not cm: continue
            conds.append(
                f"{quote_identifier(_snap_column(cm.group(1), headers))} "
                f"{_snap_op(cm.group(2))} '{_strip_quotes(cm.group(3).strip())}'"
            )
        if conds:
            out += " WHERE " + " AND ".join(conds)
    return out


# quick smoke test
_smoke = {
    "question": "test",
    "table": {"id":"t1","header":["Name","Age"],"types":["text","number"],"name":"employees"},
    "sql":   {"sel":0,"agg":3,"conds":{"column_index":[],"operator_index":[],"condition":[]}},
}
assert example_to_sql(_smoke) == 'SELECT COUNT("Name") FROM "employees"'
print("Helper functions OK.")


## 3. Tokenizer setup
Loads the Flan-T5 tokenizer and defines the batch preprocessing function.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size   : {tokenizer.vocab_size}")
print(f"pad_token_id : {tokenizer.pad_token_id}  ({tokenizer.pad_token!r})")
print(f"eos_token_id : {tokenizer.eos_token_id}  ({tokenizer.eos_token!r})")


def preprocess_batch(batch):
    inputs, targets = [], []
    for i in range(len(batch["question"])):
        ex = {
            "question": batch["question"][i],
            "table":    batch["table"][i],
            "sql":      batch["sql"][i],
        }
        inputs.append(example_to_input(ex))
        targets.append(example_to_sql(ex))
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length",
    )
    label_enc = tokenizer(
        targets, max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length",
    )
    model_inputs["labels"] = [
        [t if t != tokenizer.pad_token_id else -100 for t in row]
        for row in label_enc["input_ids"]
    ]
    return model_inputs


_sample = raw_datasets["train"][0]
_dec    = tokenizer(example_to_sql(_sample), max_length=MAX_TARGET_LENGTH, truncation=True)
assert sum(1 for t in _dec["input_ids"] if t != tokenizer.pad_token_id) > 0
print("Tokenizer OK.")

## 4. Fine-tune Flan-T5 on WikiSQL
Subsamples the data for a faster training run and fine-tunes with `Seq2SeqTrainer`. Adjust `FAST_TRAIN_SAMPLES` etc. to use the full dataset.

In [ ]:

FAST_TRAIN_SAMPLES      = 56_355
FAST_VALIDATION_SAMPLES =  8_421
FAST_TEST_SAMPLES       = 15_878
OUTPUT_DIR_FAST         = "/kaggle/working/wikisql-flan-t5-ckpts"

TRAIN_BATCH_SIZE            = 8
EVAL_BATCH_SIZE             = 8
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 32 (x n_gpus on multi-GPU)
LEARNING_RATE               = 1e-4
NUM_EPOCHS                  = 5
WARMUP_RATIO                = 0.05

fast_raw_datasets = DatasetDict({
    "train": raw_datasets["train"]
        .shuffle(seed=SEED).select(range(min(FAST_TRAIN_SAMPLES,
                                             len(raw_datasets["train"])))),
    "validation": raw_datasets["validation"]
        .shuffle(seed=SEED).select(range(min(FAST_VALIDATION_SAMPLES,
                                             len(raw_datasets["validation"])))),
    "test": raw_datasets["test"]
        .shuffle(seed=SEED).select(range(min(FAST_TEST_SAMPLES,
                                             len(raw_datasets["test"])))),
})
print("Splits:", {k: len(v) for k, v in fast_raw_datasets.items()})

fast_tokenized = fast_raw_datasets.map(
    preprocess_batch, batched=True,
    remove_columns=fast_raw_datasets["train"].column_names,
    desc="Tokenizing WikiSQL",
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id             = tokenizer.pad_token_id
model.config.eos_token_id             = tokenizer.eos_token_id
model.config.decoder_start_token_id   = tokenizer.pad_token_id
model.config.use_cache                = False
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id          = tokenizer.pad_token_id
    model.generation_config.eos_token_id           = tokenizer.eos_token_id
    model.generation_config.decoder_start_token_id = tokenizer.pad_token_id

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    label_pad_token_id=-100, pad_to_multiple_of=8,
)

_steps_ep   = math.ceil(FAST_TRAIN_SAMPLES / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
_total      = _steps_ep * NUM_EPOCHS
_warmup     = max(1, int(_total * WARMUP_RATIO))
print(f"steps/epoch={_steps_ep}  total={_total}  warmup={_warmup}")

_kwargs = dict(
    output_dir=OUTPUT_DIR_FAST,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=_warmup,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
try:
    training_args = Seq2SeqTrainingArguments(eval_strategy="epoch", **_kwargs)
except TypeError:
    training_args = Seq2SeqTrainingArguments(evaluation_strategy="epoch", **_kwargs)

try:
    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=fast_tokenized["train"],
        eval_dataset=fast_tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
except TypeError:
    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=fast_tokenized["train"],
        eval_dataset=fast_tokenized["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

print(f"\nTraining flan-t5-base on {len(fast_tokenized['train']):,} WikiSQL examples ...")
trainer.train()
trainer.model.config.use_cache = True
trainer.save_model(WIKISQL_FINAL_DIR)
tokenizer.save_pretrained(WIKISQL_FINAL_DIR)
print(f"WikiSQL model saved -> {WIKISQL_FINAL_DIR}")


## 5. Evaluate on the WikiSQL test set
Reports exact-match accuracy before and after `repair_sql` post-processing, and saves predictions to `wikisql_predictions.csv`.

In [ ]:
model.eval()
test_raw   = fast_raw_datasets["test"]
raw_preds  = []
EVAL_BATCH = 32

print(f"Evaluating on {len(test_raw):,} WikiSQL test examples ...")
for start in range(0, len(test_raw), EVAL_BATCH):
    batch_exs = [test_raw[i] for i in range(start, min(start+EVAL_BATCH, len(test_raw)))]
    prompts   = [example_to_input(e) for e in batch_exs]
    enc = tokenizer(prompts, return_tensors="pt",
                    max_length=MAX_INPUT_LENGTH, truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_length=MAX_TARGET_LENGTH, num_beams=4, early_stopping=True,
            decoder_start_token_id=model.config.decoder_start_token_id,
        )
    raw_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    if start % (EVAL_BATCH * 20) == 0 or start + EVAL_BATCH >= len(test_raw):
        print(f"  {min(start+EVAL_BATCH, len(test_raw))}/{len(test_raw)} done ...")

gold_sql = [example_to_sql(e) for e in test_raw]
rep_preds = [
    repair_sql(p,
               list(e["table"]["header"]),
               list(e["table"].get("types", ["text"]*len(e["table"]["header"]))),
               sanitize_table_name(e["table"]))
    for p, e in zip(raw_preds, test_raw)
]


def normalize_sql(sql):
    return " ".join(sql.lower().split())


em_raw = float(np.mean([normalize_sql(p) == normalize_sql(g)
                         for p, g in zip(raw_preds, gold_sql)]))
em_rep = float(np.mean([normalize_sql(p) == normalize_sql(g)
                         for p, g in zip(rep_preds, gold_sql)]))

metrics = {
    "test_examples":             len(gold_sql),
    "test_exact_match_raw":      round(em_raw, 4),
    "test_exact_match_repaired": round(em_rep, 4),
}
print(json.dumps(metrics, indent=2))

pd.DataFrame({
    "question":    [test_raw[i]["question"] for i in range(min(10, len(test_raw)))],
    "gold":        gold_sql[:10],
    "pred":        rep_preds[:10],
    "match":       [normalize_sql(g) == normalize_sql(p)
                    for g, p in zip(gold_sql[:10], rep_preds[:10])],
}).to_csv("/kaggle/working/wikisql_predictions.csv", index=False)
print("Saved wikisql_predictions.csv")
